# Bài 7: Model và cách lựa chọn

**Thời gian:** ~40 phút | **Chi phí:** $0 (không gọi API)

## Tại sao cần nhiều model khác nhau?

Giống như có nhiều loại xe cho nhiều nhu cầu khác nhau, có nhiều AI model cho nhiều tác vụ khác nhau.

So sánh với xe hơi:
- **Toyota Camry** — đáng tin cậy, giá phải chăng, phù hợp đi lại hàng ngày
- **Ferrari** — nhanh, đắt, dành cho lúc cần hiệu năng đỉnh cao
- **Xe bán tải** — không nhanh, nhưng chở được hàng nặng

AI model cũng vậy:
- **Claude Haiku** — nhanh nhất, rẻ nhất, phù hợp tác vụ đơn giản
- **Claude Sonnet** — cân bằng: nhanh, giá hợp lý, mạnh với tools và structured output
- **Claude Opus** — mạnh nhất, tốt cho suy luận phức tạp, nhưng chậm hơn và đắt hơn

Không có model nào là "tốt nhất" — tất cả phụ thuộc vào tác vụ.

Dự án của chúng ta giữ mọi thứ đơn giản: tất cả agent đều dùng **Claude Sonnet**. Một model, một API key, một nhà cung cấp.

## Đánh đổi giữa tốc độ / chi phí / chất lượng

Mỗi model cân bằng ba yếu tố. Bạn không thể có cả ba ở mức tối đa.

| Model | Tốc độ | Chi phí | Phù hợp nhất cho |
|-------|--------|---------|-------------------|
| Claude Haiku | Rất nhanh | Rất rẻ | Tác vụ đơn giản, phân loại, lọc dữ liệu |
| Claude Sonnet | Nhanh | Vừa phải | Tools, structured output, viết bài, phân tích |
| Claude Opus | Chậm | Đắt | Suy luận phức tạp, đánh giá tinh tế |

**Tại sao dùng Sonnet cho mọi thứ trong dự án?**

Sonnet nằm ở điểm cân bằng lý tưởng: đủ nhanh cho tương tác trực tiếp, đủ rẻ cho xử lý hàng loạt, và đủ mạnh cho tìm kiếm web, viết bài, tìm ảnh, và phân tích AIO. Dùng một model duy nhất giúp tránh sự phức tạp khi phải quản lý nhiều API provider khác nhau.

In [ ]:
# Hãy ước tính chi phí team cho mỗi bài viết

# Chi phí xấp xỉ trên 1M token (tính đến 2025)
models = {
    "Claude Haiku": {"input": 0.80, "output": 4.00},
    "Claude Sonnet": {"input": 3.00, "output": 15.00},
    "Claude Opus": {"input": 15.00, "output": 75.00},
}

# Lượng token sử dụng trong team (xấp xỉ)
team_steps = [
    {"step": "Team Leader", "model": "Claude Sonnet", "input_tokens": 1500, "output_tokens": 500},
    {"step": "Content Writer", "model": "Claude Sonnet", "input_tokens": 3000, "output_tokens": 3500},
    {"step": "Image Finder", "model": "Claude Sonnet", "input_tokens": 4000, "output_tokens": 500},
    {"step": "AIO Analyzer", "model": "Claude Sonnet", "input_tokens": 3000, "output_tokens": 1500},
]

total_cost = 0
print("Chi phí chi tiết cho mỗi bài viết:\n")
for step in team_steps:
    model = models[step["model"]]
    input_cost = step["input_tokens"] / 1_000_000 * model["input"]
    output_cost = step["output_tokens"] / 1_000_000 * model["output"]
    step_cost = input_cost + output_cost
    total_cost += step_cost
    print(f"  {step['step']:20s} ({step['model']:14s}): ${step_cost:.4f}")

print(f"\n  {'TỔNG CỘNG':20s}                  : ${total_cost:.4f}")
print(f"\n  Cho 100 bài viết: ${total_cost * 100:.2f}")
print(f"  Cho 1000 bài viết: ${total_cost * 1000:.2f}")

## Kiến trúc của chúng ta — Một model, ba agent

Tất cả agent trong sản phẩm đều dùng **Claude Sonnet**:

| Agent | Tools | Tại sao Sonnet? |
|-------|-------|-----------------|
| Content Writer | DataForSEO search + storage | Cần tools để tìm kiếm web. Viết bài dài tốt. |
| Image Finder | DataForSEO images + storage | Cần tools để gọi API ảnh. Đọc và cập nhật bài viết. |
| AIO Analyzer | Hàm phân tích AIO | Cần tools để gọi DataForSEO AIO API. Phân tích Google AI Overview. |
| Team Leader | — (phân công cho thành viên) | Điều phối team. Quyết định ai xử lý yêu cầu nào. |

**Tại sao không dùng model khác cho mỗi agent?**

Hoàn toàn có thể. Ví dụ, bạn có thể dùng Opus cho phân tích phức tạp hoặc Haiku cho lọc đơn giản. Nhưng với sản phẩm này:
- **Sonnet xử lý tốt mọi thứ** — tools, viết bài, phân tích
- **Một API key** — chỉ cần `ANTHROPIC_API_KEY`
- **Debug dễ hơn** — hành vi model giống nhau ở mọi nơi
- **Chi phí hợp lý** — ~$0.10-0.30 mỗi bài viết

Framework đánh đổi (tốc độ/chi phí/chất lượng) vẫn quan trọng khi bạn xây công cụ riêng. Hãy bắt đầu đơn giản với một model, rồi tối ưu sau nếu cần.

In [ ]:
# Khả năng của các model Claude

print("=== Khả Năng Của Các Model Claude ===\n")

capabilities = {
    "Claude Haiku": {"tools": True, "output_schema": True, "both_together": True, "best_at": "Tốc độ + chi phí thấp"},
    "Claude Sonnet": {"tools": True, "output_schema": True, "both_together": True, "best_at": "Cân bằng (lựa chọn của chúng ta)"},
    "Claude Opus": {"tools": True, "output_schema": True, "both_together": True, "best_at": "Suy luận phức tạp"},
}

for model, caps in capabilities.items():
    check = lambda x: "Yes" if x else "No"
    print(f"  {model:15s}  tools: {check(caps['tools']):3s}  schema: {check(caps['output_schema']):3s}  kết hợp: {check(caps['both_together']):3s}  | {caps['best_at']}")

print("\n=== Yêu Cầu Của Các Agent Trong Team ===\n")

agents = [
    {"name": "Content Writer", "needs_tools": True, "needs_schema": False, "model": "Claude Sonnet"},
    {"name": "Image Finder", "needs_tools": True, "needs_schema": False, "model": "Claude Sonnet"},
    {"name": "AIO Analyzer", "needs_tools": True, "needs_schema": False, "model": "Claude Sonnet"},
    {"name": "Team Leader", "needs_tools": False, "needs_schema": False, "model": "Claude Sonnet"},
]

for agent in agents:
    needs = []
    if agent["needs_tools"]: needs.append("tools")
    if agent["needs_schema"]: needs.append("schema")
    needs_str = ", ".join(needs) if needs else "không cần gì"
    print(f"  {agent['name']:18s} cần: {needs_str:10s} \u2192 {agent['model']}")

## Embedding — Một Dạng AI Khác

Không phải mọi output của AI đều là văn bản. **Embedding** biến văn bản thành danh sách các con số thể hiện ý nghĩa.

Tương tự tọa độ GPS. "Tokyo" và "Thủ đô Nhật Bản" nằm gần nhau trên bản đồ ý nghĩa. "Làm bánh mì" thì ở rất xa.

```
"SEO"                        → [0.82, 0.15, 0.91, ...]
"search engine optimization" → [0.80, 0.17, 0.89, ...]  ← gần nhau!
"baking bread"               → [0.12, 0.88, 0.05, ...]  ← rất xa
```

Embedding được dùng ở đâu:
- **Tìm kiếm ngữ nghĩa** — tìm bài viết tương tự với truy vấn (không chỉ khớp từ khóa)
- **Phân cụm nội dung** — nhóm các bài viết tương tự lại với nhau
- **RAG** (Retrieval-Augmented Generation) — tìm tài liệu liên quan để đưa cho LLM

Chúng ta không dùng embedding trong pipeline này, nhưng đây là khái niệm quan trọng cho các công cụ AI sau này.

In [ ]:
# Demo embedding đơn giản (embedding thật dùng hàng trăm chiều)
# Đây chỉ minh họa KHÁI NIỆM — số thật đến từ embedding model

fake_embeddings = {
    "SEO optimization": [0.9, 0.8, 0.1, 0.2],
    "search engine ranking": [0.85, 0.75, 0.15, 0.25],
    "content marketing": [0.6, 0.5, 0.7, 0.3],
    "baking sourdough bread": [0.1, 0.1, 0.2, 0.9],
}

def similarity(a, b):
    """Tích vô hướng đơn giản (hệ thống thật dùng cosine similarity)"""
    return sum(x * y for x, y in zip(a, b))

print("Điểm tương đồng (cao hơn = giống nhau hơn):\n")
base = "SEO optimization"
base_emb = fake_embeddings[base]

for text, emb in fake_embeddings.items():
    if text != base:
        score = similarity(base_emb, emb)
        bar = "\u2588" * int(score * 10)
        print(f"  \"{base}\" vs \"{text}\"")
        print(f"    Điểm: {score:.2f}  {bar}\n")

## Chọn model phù hợp — Hướng dẫn quyết định

Khi bạn cần quyết định dùng model nào cho một agent, hãy đi theo sơ đồ sau:

1. **Agent có cần tools không** (tìm kiếm web, API)?
   - Có → Phải dùng model hỗ trợ tools (tất cả model Claude đều hỗ trợ)
2. **Agent có cần structured output không** (output_schema)?
   - Có → Phải dùng model hỗ trợ output_schema
3. **Tác vụ đơn giản không** (phân loại, lọc, trả lời có/không)?
   - Có → Cân nhắc **Haiku** (rẻ nhất, nhanh nhất)
4. **Tác vụ yêu cầu suy luận phức tạp** (lập kế hoạch, phân tích nhiều bước)?
   - Có → Cân nhắc **Opus** (mạnh nhất, nhưng đắt)
5. **Mặc định** → **Claude Sonnet** (cân bằng tốt nhất giữa tốc độ, chi phí và khả năng)

Dự án của chúng ta theo quy tắc 5: Sonnet cho mọi thứ. Khi xây công cụ riêng, bạn có thể dùng Haiku cho tiền xử lý hoặc Opus cho các quyết định phức tạp.

In [ ]:
# Bài tập: Ghép agent với model!
# Với mỗi tình huống, bạn sẽ chọn model nào và tại sao?

scenarios = [
    {
        "name": "Email Writer Agent",
        "description": "Viết email marketing từ brief",
        "needs_tools": False,
        "needs_schema": False,
        "priority": "Chất lượng viết + chi phí thấp",
    },
    {
        "name": "Competitor Analyzer",
        "description": "Tìm kiếm web và trả về dữ liệu đối thủ có cấu trúc",
        "needs_tools": True,
        "needs_schema": True,
        "priority": "Độ chính xác",
    },
    {
        "name": "Content Classifier",
        "description": "Phân loại bài viết theo chủ đề (trả về nhãn danh mục)",
        "needs_tools": False,
        "needs_schema": True,
        "priority": "Tốc độ và chi phí",
    },
]

print("=== Bài Tập Chọn Model ===\n")
for s in scenarios:
    print(f"Agent: {s['name']}")
    print(f"  Tác vụ: {s['description']}")
    print(f"  Cần tools: {'Có' if s['needs_tools'] else 'Không'}")
    print(f"  Cần schema: {'Có' if s['needs_schema'] else 'Không'}")
    print(f"  Ưu tiên: {s['priority']}")
    print(f"  Lựa chọn của bạn: _______________")
    print()

print("Hãy suy nghĩ câu trả lời, rồi xem đáp án gợi ý bên dưới!")

In [ ]:
# Đáp án gợi ý:

answers = {
    "Email Writer Agent": "Claude Sonnet \u2014 chất lượng viết tốt, chi phí vừa phải. Không cần tools hay schema, nên Haiku cũng được nếu ưu tiên tiết kiệm.",
    "Competitor Analyzer": "Claude Sonnet \u2014 cần cả tools LẪN schema. Sonnet xử lý tốt cả hai với chi phí hợp lý.",
    "Content Classifier": "Claude Haiku \u2014 tác vụ đơn giản, cần schema nhưng nhẹ. Ưu tiên tốc độ và chi phí, Haiku là rẻ nhất.",
}

print("=== Đáp Án Gợi Ý ===\n")
for agent, answer in answers.items():
    print(f"  {agent}: {answer}\n")

## Tổng kết Module 2 — Hiểu về AI

| Bài | Chủ đề | Điểm mấu chốt |
|-----|--------|----------------|
| Bài 5 | Cách LLM hoạt động | LLM dự đoán văn bản từng token một. Chúng có knowledge cutoff và có thể bịa thông tin (hallucinate). |
| Bài 6 | Prompt và Context | Prompt tốt = Vai trò + Tác vụ + Ràng buộc + Ví dụ. `instructions` chính là system prompt. |
| Bài 7 | Model và cách lựa chọn | Mỗi model phù hợp với mỗi tác vụ khác nhau. Dự án của chúng ta dùng Sonnet cho mọi thứ để giữ đơn giản. |

Bây giờ bạn đã hiểu **TẠI SAO** các agent hoạt động theo cách đó. Trong Module 3, bạn sẽ **XÂY DỰNG** chúng.

**Module tiếp theo:** Module 3 — Xây Dựng Agent. Bạn sẽ tạo agent AI đầu tiên, trang bị tools cho nó, khiến nó trả về dữ liệu có cấu trúc, nối chuỗi các agent lại, và xây dựng một mini pipeline!

## Bài tập

Không cần code — bài tập tư duy:

Agency SEO của bạn muốn xây một công cụ AI mới: **"Content Audit Agent"** có khả năng:
- Crawl URL website để đọc nội dung hiện có (cần tool)
- Phân tích nội dung để tìm vấn đề SEO
- Trả về báo cáo có cấu trúc với điểm số và khuyến nghị (cần schema)
- Phải đủ rẻ để chạy trên hàng trăm trang

**Câu hỏi:**
1. Bạn sẽ chọn model nào? Tại sao?
2. Có thể dùng Haiku cho việc này không? Tại sao có hoặc tại sao không?
3. Làm sao để giữ chi phí thấp khi xử lý hàng trăm trang?

<details>
<summary>Nhấn để xem đáp án gợi ý</summary>

1. **Claude Sonnet** — cần cả tools (crawl web) lẫn structured output (schema báo cáo), và Sonnet đủ rẻ để dùng hàng loạt.
2. **Có thể cho một phần** — Haiku có thể làm bước lọc sơ bộ ("trang này có vấn đề SEO không?"), nhưng phân tích chi tiết và tạo báo cáo cần suy luận sâu hơn của Sonnet.
3. Dùng **Haiku** cho bước lọc sơ bộ (kiểm tra nhanh xem trang có cần audit kỹ không), rồi chỉ gửi những trang cần phân tích sâu tới Sonnet. Ngoài ra, giữ prompt ngắn gọn để giảm thiểu input token.

</details>

## Sẵn sàng cho Module 3? — Checkpoint

Module 3 là lúc thực hành thật: bạn sẽ tạo AI agent gọi API Claude.

- **Gọi API thật** — mỗi ô tốn một khoản nhỏ (~$0.01-0.10)
- **Cần API key** — `ANTHROPIC_API_KEY` phải được đặt trong file `.env`
- **Cần internet** — agent gọi API bên ngoài

Chạy ô bên dưới để kiểm tra.

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()

# Check: Anthropic API key
print("Anthropic API key (cho Claude)...")
key = os.getenv("ANTHROPIC_API_KEY", "")
if len(key) > 10:
    print(f"   PASS - Tìm thấy key ({key[:8]}...)")
    print("\nBạn đã sẵn sàng cho Module 3!")
else:
    print("   FAIL - ANTHROPIC_API_KEY chưa được đặt")
    print("   Cách sửa: Thêm key vào file .env trong thư mục gốc của dự án")
    print("   Lấy key tại: https://console.anthropic.com")
    print("\nHãy sửa lỗi trên trước khi bắt đầu Module 3.")